LipidTOX droplets (red, idx0) · LAMP1-RFP lysosomes (orange, idx1) · galectin-GFP (green, idx2) · Hoechst nuclei (blue, idx3)

## Imports
Install dependencies first.

In [ ]:
%matplotlib inline
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, ListedColormap

import czifile
from scipy import ndimage as ndi
from skimage import exposure, morphology
from skimage.filters import threshold_otsu, gaussian
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects, disk, binary_dilation
from skimage.segmentation import relabel_sequential, watershed

## Parameters & segmentation method

In [ ]:
BASE_DIR = "."
IMAGE_FOLDER_NAMES = [
    "M17D_52926_siRNA_galectinGFP_LAMP1RFP_LipidTOXfr_hoechst",
    "M17D_60426_siRNA_galectinGFP_LAMP1RFP_LipidTOXfr_hoechst",
    "M17D_60526_siRNA_OA_Fed_galectinGFP_LAMP1RFP_LipidTOXfr_hoechst",
    "M17D_60926_siRNA_OA_Fed_galectinGFP_LAMP1RFP_LipidTOXfr_Hoechst",
    "M17D_61126_siRNA_galectinGFP_LAMP1RFP_LipidTOXfr_hoechst",
    "M17D_61826_siRNA_galectinGFP_LAMP1RFP_LipidTOXfr_hoechst",
    "M17D_61826_siRNA_OA_Fed_galectinGFP_LAMP1RFP_LipidTOXfr_Hoechst",
]


def resolve_image_folder(name):
    """Return the first folder that exists for name."""
    for cand in (os.path.join(BASE_DIR, name), os.path.join(BASE_DIR, "images", name)):
        if os.path.isdir(cand):
            return cand
    return os.path.join(BASE_DIR, name)


IMAGE_FOLDERS = [resolve_image_folder(n) for n in IMAGE_FOLDER_NAMES]

FILE_EXTENSION = "*.czi"
RESULTS_FOLDER = "Results"
MASK_FOLDER = os.path.join(RESULTS_FOLDER, "masks")
CSV_SUFFIX = "galectin_LAMP1_lipidtox_analysis.csv"  # final CSV suffix
SHOW_PLOTS = True  # skip QC panels on long runs

# Channel order: red, orange, green, blue
CHANNEL_ORDER = {"red": 0, "orange": 1, "green": 2, "blue": 3}

USE_CELLPOSE = True
USE_GPU = True

# Cell segmentation: Cellpose cyto3 on orange + red
CYTO_SEG_CHANNELS = ["orange", "red"]
CYTO_DIAMETER = 110  # larger keeps big cells whole
CYTO_FLOW_THRESH = 0.8  # more lenient
CYTO_PROB_THRESH = -3.0  # catches dimmer cells
CELL_MIN_AREA = 300  # drop tiny objects

# Hybrid fill: recover missed cells from nuclei
CELL_FILL_FG_THRESH = 0.06  # composite cutoff for cell foreground
CELL_GROW_DIST = 70  # max growth from a nucleus

# Nucleus segmentation: permissive on blue
NUC_DIAMETER = 50
NUC_FLOW_THRESH = 0.9  # permissive
NUC_PROB_THRESH = -6.0  # very sensitive
NUC_MIN_AREA = 30
NUC_INTENSITY_FLOOR = 0.06  # anything above this is nucleus too

# Green-positive cell: threshold on green
GREEN_FILL_SIGMA = 2.0  # smooth green before thresholding
GREEN_FILL_THRESH = 0.10  # green above this counts
GREEN_FILL_FRAC = 0.50  # green-positive cutoff

# Fallback if Cellpose is off
SEG_SMOOTH_SIGMA = 10
SEG_FG_FLOOR = 0.05
SEG_GREEN_BOOST = 3.0
NUCLEUS_MIN_SIZE = 30
NUCLEUS_FLOOR = 0.05

# Inclusion analysis
GREEN_TOPHAT_RADIUS = 8
GREEN_CONTRAST_THRESH = 0.010
GREEN_MIN_BRIGHTNESS = 0.05
GREEN_MIN_SIZE = 4
GREEN_MAX_SIZE = 400
GREEN_LOCAL_ZSCORE = 1.5
GREEN_LOCAL_PCTL = 85

# Red lipid droplets
RED_TOPHAT_RADIUS = 12
RED_PCTL_THRESH = 99.0
RED_MIN_SIZE = 6
RED_MAX_SIZE = 1500

# Orange lysosomes
ORANGE_TOPHAT_RADIUS = 12
ORANGE_PCTL_THRESH = 98.5
ORANGE_MIN_SIZE = 6
ORANGE_MAX_SIZE = 1500

# Bleed-through: red to orange
BLEED_CONTAINMENT_THRESH = 0.90
BLEED_PROTECT_GREEN = True
TOUCH_DILATION = 1
STRUCTURE_CLOSE_RADIUS = 1

os.makedirs(RESULTS_FOLDER, exist_ok=True)
os.makedirs(MASK_FOLDER, exist_ok=True)

cyto_model = nuc_model = None
if USE_CELLPOSE:
    from cellpose import models

## Loading & normalization

In [12]:
def normalize_image(image):
    """Min-max normalize to float64 [0, 1].  Used by the INCLUSION ANALYSIS
    (green dots, lipids, lysosomes) — do not change."""
    img = image.astype(np.float64)
    mn, mx = img.min(), img.max()
    return np.zeros_like(img) if mx == mn else (img - mn) / (mx - mn)


def pct_norm(image, lo=1.0, hi=99.5):
    """Percentile contrast-stretch to float32 [0, 1].  Used for SEGMENTATION.

    Min-max normalization fails on these images because single saturated pixels
    (raw max ~65535) crush the real cell signal to ~0.01. Clipping at the 1st/
    99.5th percentiles keeps the cytoplasm/nucleus signal in a usable range so
    Cellpose can actually see the cells."""
    image = image.astype(np.float32)
    p_lo, p_hi = np.percentile(image, (lo, hi))
    if p_hi <= p_lo:
        return np.zeros_like(image)
    return np.clip((image - p_lo) / (p_hi - p_lo), 0, 1)


def boost_contrast_visual(image):
    """Percentile stretch for DISPLAY ONLY."""
    p_lo, p_hi = np.percentile(image, (1, 99.5))
    return exposure.rescale_intensity(image, in_range=(p_lo, p_hi))


def load_four_channels(filepath):
    """Load 4-channel .czi -> dict {'red','orange','green','blue'} per CHANNEL_ORDER."""
    try:
        czi = czifile.CziFile(filepath)
        img = np.squeeze(czi.asarray())
    except Exception as e:
        print(f"  [ERROR] File load failed: {e}")
        return None
    if img.ndim == 3 and 4 in img.shape:
        ch_axis = int(np.where(np.array(img.shape) == 4)[0][0])
        img = np.moveaxis(img, ch_axis, 0)
    else:
        print(f"  [ERROR] Expected a 4-channel image, got shape {img.shape}")
        return None
    return {name: img[idx] for name, idx in CHANNEL_ORDER.items()}

## Mask functions

In [ ]:


def drop_small_labels(labels, min_area):
    """Remove labels smaller than min_area and relabel 1..N."""
    labels = labels.astype(np.int32)
    counts = np.bincount(labels.ravel())
    small = np.flatnonzero(counts < min_area)
    small = small[small != 0]  # keep background
    if small.size:
        labels[np.isin(labels, small)] = 0
    return relabel_sequential(labels)[0]


def cyto_composite(seg):
    """Max of the cell-segmentation channels."""
    return np.maximum.reduce([seg[c] for c in CYTO_SEG_CHANNELS])


def segment_whole_cells_cellpose(seg):
    """Cell mask from Cellpose cyto3 on orange + red."""
    composite = cyto_composite(seg)
    masks, _, _, _ = cyto_model.eval(
        composite, diameter=CYTO_DIAMETER, channels=[0, 0],
        flow_threshold=CYTO_FLOW_THRESH, cellprob_threshold=CYTO_PROB_THRESH,
        normalize=True, resample=True,
    )
    cleaned = masks.astype(np.int32)
    for rid in range(1, int(cleaned.max()) + 1):
        if (cleaned == rid).sum() < CELL_MIN_AREA:
            cleaned[cleaned == rid] = 0
    final, _, _ = relabel_sequential(cleaned)
    print(f"    Cells (cyto3 orange+red, composite-only): {final.max()}")
    return final


def segment_nucleus(seg):
    """Return the nucleus mask and labeled nuclei."""
    blue = seg["blue"]
    masks, _, _, _ = nuc_model.eval(
        (blue * 255).astype(np.uint8), diameter=NUC_DIAMETER, channels=[0, 0],
        flow_threshold=NUC_FLOW_THRESH, cellprob_threshold=NUC_PROB_THRESH,
        normalize=True, resample=True,
    )
    cp_nuc = masks > 0
    labeled_nuclei = drop_small_labels(masks, NUC_MIN_AREA)  # nuclei for seeding

    # Catch bright blue pixels the model missed
    bsm = gaussian(blue, sigma=2)
    try:
        thr = max(threshold_otsu(bsm) * 0.5, NUC_INTENSITY_FLOOR)
    except Exception:
        thr = NUC_INTENSITY_FLOOR
    inten = bsm > thr
    inten = morphology.binary_closing(inten, morphology.disk(2))
    inten = ndi.binary_fill_holes(inten)
    inten = remove_small_objects(inten, min_size=NUC_MIN_AREA)

    nucleus = cp_nuc | inten
    print(f"    Nucleus (cellpose {int(masks.max())} + intensity union): "
          f"{int(nucleus.sum())} px")
    return nucleus, labeled_nuclei


def fill_missed_cells_from_nuclei(cyto_cells, nuclei, composite):
    """Recover missed cells from nuclei."""
    cells = cyto_cells.astype(np.int32).copy()
    n_cyto = int(cells.max())

    # Existing cells plus one new label per missed nucleus
    seeds = cells.copy()
    next_label = n_cyto + 1
    for nid in np.unique(nuclei):
        if nid == 0:
            continue
        nuc = nuclei == nid
        if (cells[nuc] > 0).mean() < 0.5:  # nucleus not inside a cell
            seeds[nuc & (cells == 0)] = next_label
            next_label += 1
    n_added = next_label - 1 - n_cyto
    if n_added == 0:
        print(f"    Hybrid cells: {n_cyto} (+0 recovered)")
        return cells

    # Grow new seeds within signal, up to a max distance
    dist_from_seed = ndi.distance_transform_edt(seeds == 0)
    foreground = composite > CELL_FILL_FG_THRESH
    foreground = ndi.binary_fill_holes(morphology.closing(foreground, disk(3))) | (seeds > 0)
    foreground = foreground & (dist_from_seed <= CELL_GROW_DIST)
    grown = watershed(
        np.where(foreground, dist_from_seed, dist_from_seed.max() + 1),
        markers=seeds, mask=foreground
    )
    new_pixels = (grown > n_cyto) & (cyto_cells == 0)  # never overwrite cyto3 cells
    cells[new_pixels] = grown[new_pixels]
    final, _, _ = relabel_sequential(cells)
    print(f"    Hybrid cells: {int(final.max())} (+{n_added} recovered)")
    return final


def make_nucleus_mask(blue):
    """Intensity-only nucleus fallback."""
    bsm = gaussian(blue, sigma=SEG_SMOOTH_SIGMA)
    bsm = normalize_image(bsm)
    nuc = bsm > SEG_FG_FLOOR
    nuc = morphology.binary_closing(nuc, morphology.disk(2))
    nuc = remove_small_objects(nuc, min_size=NUCLEUS_MIN_SIZE)
    return nuc


def segment_cells_intensity(green, blue):
    """Intensity-based cell fallback."""
    g = normalize_image(gaussian(green, sigma=SEG_SMOOTH_SIGMA))
    b = normalize_image(gaussian(blue, sigma=SEG_SMOOTH_SIGMA))
    score = np.maximum(g * SEG_GREEN_BOOST, b)
    fg = score > SEG_FG_FLOOR
    fg = morphology.binary_closing(fg, morphology.disk(3))
    fg = remove_small_objects(fg, min_size=CELL_MIN_AREA)
    labeled = label(fg)
    cleaned = np.zeros_like(labeled, dtype=np.int32)
    for rid in range(1, int(labeled.max()) + 1):
        if (labeled == rid).sum() < CELL_MIN_AREA:
            cleaned[labeled == rid] = 0
    final, _, _ = relabel_sequential(cleaned)
    print(f"    Cells (intensity): {final.max()}")
    return final


def bridge_small_gaps(mask, radius=STRUCTURE_CLOSE_RADIUS):
    if radius <= 0:
        return mask
    return morphology.binary_closing(mask, morphology.disk(radius))


# Inclusion analysis
def detect_green_dots(green_norm, labeled_cells):
    """Cell-aware galectin puncta."""
    tophat = morphology.white_tophat(green_norm, morphology.disk(GREEN_TOPHAT_RADIUS))
    candidate = ((tophat > GREEN_CONTRAST_THRESH) &
                 (green_norm > GREEN_MIN_BRIGHTNESS) & (labeled_cells > 0))
    candidate = remove_small_objects(candidate, min_size=GREEN_MIN_SIZE)
    final = np.zeros_like(candidate, dtype=bool)
    for cid in range(1, int(labeled_cells.max()) + 1):
        cell_mask = labeled_cells == cid
        vals = green_norm[cell_mask]
        if vals.size == 0:
            continue
        local_thr = max(GREEN_MIN_BRIGHTNESS,
                        float(vals.mean()) + GREEN_LOCAL_ZSCORE * float(vals.std() or 1e-6),
                        float(np.percentile(vals, GREEN_LOCAL_PCTL)))
        final[candidate & cell_mask & (green_norm >= local_thr)] = True
    final = bridge_small_gaps(final)
    final = remove_small_objects(final, min_size=GREEN_MIN_SIZE)
    lbl = label(final)
    for p in regionprops(lbl):
        if p.area > GREEN_MAX_SIZE:
            final[lbl == p.label] = False
    return final


def detect_bright_puncta(channel_norm, tophat_radius, pctl_thresh, min_size, max_size):
    tophat = morphology.white_tophat(channel_norm, morphology.disk(tophat_radius))
    nonzero = tophat[tophat > 0]
    if nonzero.size == 0:
        return np.zeros_like(channel_norm, dtype=bool)
    mask = tophat > np.percentile(nonzero, pctl_thresh)
    mask = bridge_small_gaps(mask)
    mask = remove_small_objects(mask, min_size=min_size)
    lbl = label(mask)
    for p in regionprops(lbl):
        if p.area > max_size:
            mask[lbl == p.label] = False
    return mask


def remove_bleedthrough(orange_mask, red_mask, green_dots=None,
                        thresh=BLEED_CONTAINMENT_THRESH, protect_green=BLEED_PROTECT_GREEN):
    orange_lbl = label(orange_mask)
    red_lbl = label(red_mask)
    red_area = {p.label: p.area for p in regionprops(red_lbl)}
    corrected = orange_mask.copy()
    bleed = np.zeros_like(orange_mask, dtype=bool)
    for o in regionprops(orange_lbl):
        r0, c0, r1, c1 = o.bbox
        o_crop = orange_lbl[r0:r1, c0:c1] == o.label
        red_crop = red_lbl[r0:r1, c0:c1]
        overlap = np.unique(red_crop[o_crop]); overlap = overlap[overlap > 0]
        if overlap.size == 0:
            continue
        if protect_green and green_dots is not None and np.any(o_crop & green_dots[r0:r1, c0:c1]):
            continue
        red_union = np.isin(red_crop, overlap)
        frac_orange_in_red = (o_crop & red_union).sum() / o.area
        red_in_orange = any(((red_crop == rl) & o_crop).sum() / red_area[rl] >= thresh
                            for rl in overlap)
        if frac_orange_in_red >= thresh or red_in_orange:
            corr_crop = corrected[r0:r1, c0:c1]; bleed_crop = bleed[r0:r1, c0:c1]
            corr_crop[o_crop] = False; bleed_crop[o_crop] = True
            corrected[r0:r1, c0:c1] = corr_crop; bleed[r0:r1, c0:c1] = bleed_crop
    return corrected, bleed


print("Mask functions defined.")

## Overlap helpers

In [ ]:
def count_dots_overlapping(dot_mask_in_cell, target_mask):
    """Count dots that touch target_mask."""
    lbl = label(dot_mask_in_cell)
    n, area = 0, 0
    for p in regionprops(lbl):
        if np.any((lbl == p.label) & target_mask):
            n += 1; area += p.area
    return n, area


def triple_positive_overlap(green_dots, orange_mask, red_mask, cell_mask):
    """Count triple-positive clumps and total overlap area in one cell."""
    triple = green_dots & orange_mask & red_mask & cell_mask
    if not triple.any():
        return 0, 0, triple

    union = (green_dots | orange_mask | red_mask) & cell_mask
    union_lbl = label(union)

    n_structures = 0
    valid_union = np.zeros_like(union, dtype=bool)
    for p in regionprops(union_lbl):
        comp = union_lbl == p.label
        has_green = np.any(comp & green_dots)
        has_orange = np.any(comp & orange_mask)
        has_red = np.any(comp & red_mask)
        has_true_triple = np.any(comp & triple)
        if has_green and has_orange and has_red and has_true_triple:
            n_structures += 1
            valid_union[comp] = True

    triple_valid = triple & valid_union
    return int(n_structures), int(triple_valid.sum()), triple_valid


def triple_positive_touching(green_dots, orange_mask, red_mask, cell_mask,
                             dilation=TOUCH_DILATION):
    """Count orange blobs touching both red and green."""
    lbl = label(orange_mask & cell_mask)
    out_mask = np.zeros_like(orange_mask, dtype=bool)
    n = 0
    H, W = orange_mask.shape
    for p in regionprops(lbl):
        r0, c0, r1, c1 = p.bbox
        r0d, c0d = max(r0 - dilation, 0), max(c0 - dilation, 0)
        r1d, c1d = min(r1 + dilation, H), min(c1 + dilation, W)
        blob = lbl[r0d:r1d, c0d:c1d] == p.label
        grown = binary_dilation(blob, footprint=disk(dilation))
        if (np.any(grown & red_mask[r0d:r1d, c0d:c1d]) and
                np.any(grown & green_dots[r0d:r1d, c0d:c1d])):
            n += 1
            out_crop = out_mask[r0d:r1d, c0d:c1d]
            out_crop[blob] = True
            out_mask[r0d:r1d, c0d:c1d] = out_crop
    return n, out_mask


print("Helper + mask functions defined.")

Helper + mask functions defined.


## Per-cell analysis

In [ ]:
def analyze_cell(cid, labeled_cells, masks, raw):
    """Return all per-cell measurements for one green-positive cell."""
    cell_mask = labeled_cells == cid
    cyto_mask = cell_mask & ~masks["nucleus"]  # cell minus nucleus
    green_raw = raw["green"]

    # Limit all masks to this cell first
    green_dots = masks["green_dots"] & cell_mask
    orange_in = masks["orange"] & cell_mask  # corrected lysosomes
    red_in = masks["red"] & cell_mask  # lipid droplets

    # Galectin dots: count, area, intensity
    n_green_dots = int(label(green_dots).max())
    green_dot_area = int(green_dots.sum())
    green_dot_mfi = float(green_raw[green_dots].mean()) if green_dots.any() else 0.0

    # Whole-cell galectin intensity
    green_cyto_mfi = float(green_raw[cyto_mask].mean()) if cyto_mask.any() else 0.0
    green_entire_mfi = float(green_raw[cell_mask].mean()) if cell_mask.any() else 0.0

    # Green dots on orange lysosomes
    n_green_on_orange, area_green_on_orange = count_dots_overlapping(green_dots, orange_in)

    # Triple-positive counts
    n_triple_overlap, triple_overlap_area, _ = triple_positive_overlap(
        green_dots, orange_in, red_in, cell_mask)
    n_triple_touch, _ = triple_positive_touching(
        green_dots, orange_in, red_in, cell_mask)

    return {
        "Cell_ID": cid,
        "Cell_Area_px": int(cell_mask.sum()),
        "Cytoplasm_Area_px": int(cyto_mask.sum()),
        "Nucleus_Area_px": int((masks["nucleus"] & cell_mask).sum()),
        "N_Green_Dots": n_green_dots,
        "Green_Dot_Area_px": green_dot_area,
        "Green_Dot_MFI": round(green_dot_mfi, 3),
        "Green_WholeCell_MFI": round(green_cyto_mfi, 3),
        "Green_EntireCell_MFI": round(green_entire_mfi, 3),
        "N_GreenDots_on_Orange": n_green_on_orange,
        "GreenDots_on_Orange_Area": area_green_on_orange,
        "N_Lysosomes": int(label(orange_in).max()),
        "Lysosome_Area_px": int(orange_in.sum()),
        "N_Lipid_Droplets": int(label(red_in).max()),
        "LipidDroplet_Area_px": int(red_in.sum()),
        "N_TriplePositive_Overlap": n_triple_overlap,
        "TriplePositive_Overlap_Area_px": triple_overlap_area,
        "N_TriplePositive_Touching": n_triple_touch,
    }

## Per-image segmentation

In [ ]:
cmap_g = LinearSegmentedColormap.from_list("G", [(0, 0, 0), (0, 1, 0)])
cmap_r = LinearSegmentedColormap.from_list("R", [(0, 0, 0), (1, 0, 0)])
cmap_o = LinearSegmentedColormap.from_list("O", [(0, 0, 0), (1, 0.55, 0)])
cmap_b = LinearSegmentedColormap.from_list("B", [(0, 0, 0), (0, 0.4, 1)])


def show_mask(ax, mask, title, color="cyan"):
    H, W = mask.shape
    ax.imshow(np.zeros((H, W)), cmap="gray", vmin=0, vmax=1)
    if mask.any():
        ax.imshow(np.ma.masked_where(~mask, np.ones((H, W))),
                  cmap=ListedColormap([color]), interpolation="none")
    ax.set_title(title, color="white", fontsize=8)


def show_mask_with_overlay(ax, base_mask, overlay_mask, title,
                           base_color="orange", overlay_color="magenta"):
    H, W = base_mask.shape
    ax.imshow(np.zeros((H, W)), cmap="gray", vmin=0, vmax=1)
    if base_mask.any():
        ax.imshow(np.ma.masked_where(~base_mask, np.ones((H, W))),
                  cmap=ListedColormap([base_color]), interpolation="none")
    if overlay_mask.any():
        ax.imshow(np.ma.masked_where(~overlay_mask, np.ones((H, W))),
                  cmap=ListedColormap([overlay_color]), interpolation="none", alpha=0.85)
    ax.set_title(title, color="white", fontsize=8)


def save_image_masks(img_mask_dir, labeled_cells, masks, green_positive_ids):
    """Save masks for one image in its own folder."""
    os.makedirs(img_mask_dir, exist_ok=True)
    np.save(os.path.join(img_mask_dir, "labeled_cells.npy"), labeled_cells)
    np.save(os.path.join(img_mask_dir, "nucleus.npy"), masks["nucleus"])
    np.save(os.path.join(img_mask_dir, "green_dots.npy"), masks["green_dots"])
    np.save(os.path.join(img_mask_dir, "red_lipid.npy"), masks["red"])
    np.save(os.path.join(img_mask_dir, "orange_lysosome.npy"), masks["orange"])
    np.save(os.path.join(img_mask_dir, "bleedthrough.npy"), masks["bleed"])
    np.save(os.path.join(img_mask_dir, "green_positive_ids.npy"),
            np.array(green_positive_ids, dtype=np.int32))


def show_fourteen_panels(filename, raw, labeled_cells, green_positive_ids,
                         nucleus_mask, green_dots, red_mask, orange_raw,
                         orange_mask, bleed_mask, cyto3_cells):
    """Build the 5x4 QC figure."""
    m1_cells = labeled_cells > 0
    m2_greenpos = np.isin(labeled_cells, green_positive_ids)
    m3_nucleus = nucleus_mask
    m4_cytoplasm = m2_greenpos & ~nucleus_mask
    m12_go = np.zeros_like(green_dots)
    _gl = label(green_dots)
    for p in regionprops(_gl):
        blob = _gl == p.label
        if np.any(blob & orange_mask):
            m12_go[blob] = True
    m13_triple = green_dots & orange_mask & red_mask
    _, m14_triple_touch = triple_positive_touching(green_dots, orange_mask, red_mask, m1_cells)

    fig, ax = plt.subplots(5, 4, figsize=(22, 26))
    fig.patch.set_facecolor("black")
    fig.suptitle(filename, color="white", fontsize=11, y=0.998)

    for a, im, cm, t in [
        (ax[0, 0], boost_contrast_visual(raw["green"]), cmap_g, "Green (galectin)"),
        (ax[0, 1], boost_contrast_visual(raw["orange"]), cmap_o, "Orange (LAMP1)"),
        (ax[0, 2], boost_contrast_visual(raw["red"]), cmap_r, "Red (LipidTOX)"),
        (ax[0, 3], boost_contrast_visual(raw["blue"]), cmap_b, "Blue (Hoechst)")]:
        a.imshow(im, cmap=cm); a.set_title(t, color="white", fontsize=9)

    ax[1, 0].imshow(np.where(m1_cells, labeled_cells, np.nan), cmap="nipy_spectral",
                    interpolation="none", vmin=1)
    n_cyto3 = int(cyto3_cells.max()) if cyto3_cells is not None else int(labeled_cells.max())
    n_filled = int(labeled_cells.max()) - n_cyto3
    ax[1, 0].set_title(f"1. Cell mask — {labeled_cells.max()} cells "
                       f"(cyto3 {n_cyto3} + filled {n_filled})", color="white", fontsize=8)
    ax[1, 1].imshow(np.where(m2_greenpos, labeled_cells, np.nan), cmap="nipy_spectral",
                    interpolation="none", vmin=1)
    ax[1, 1].set_title(f"2. Green-positive cells — {len(green_positive_ids)}", color="white", fontsize=8)

    show_mask(ax[1, 2], m3_nucleus, "3. Nucleus mask", "deepskyblue")
    show_mask(ax[1, 3], m4_cytoplasm, "4. Cytoplasm (cell - nucleus)", "violet")
    show_mask(ax[2, 0], m2_greenpos, "5. Green whole-cell galectin", "lime")
    show_mask(ax[2, 1], green_dots, "6. Green bright-dot mask", "lime")
    show_mask(ax[2, 2], red_mask, "7. Red lipid droplet mask", "red")
    show_mask(ax[2, 3], orange_raw, "8. Orange lysosome (raw)", "orange")
    show_mask_with_overlay(ax[3, 0], orange_mask, bleed_mask,
                           "9. Orange + bleedthrough map", "orange", "magenta")
    show_mask(ax[3, 1], bleed_mask, "10. Bleedthrough mask", "magenta")
    show_mask(ax[3, 2], orange_mask, "11. Orange lysosome-dot (final)", "orange")
    show_mask(ax[3, 3], m12_go, "12. Green-Orange overlap", "yellow")
    show_mask(ax[4, 0], m13_triple, "13. Triple-overlap mask", "white")
    show_mask(ax[4, 1], m14_triple_touch, "14. Triple-positive (touching)", "cyan")
    ax[4, 2].axis("off"); ax[4, 3].axis("off")

    for a in ax.ravel():
        a.set_facecolor("black")
        if a.get_title() == "":
            a.axis("off")
        a.set_xticks([]); a.set_yticks([])
        for spine in a.spines.values():
            spine.set_visible(True); spine.set_color("white"); spine.set_linewidth(0.6)
    plt.tight_layout()
    plt.show()


def segment_one_image(filepath, mask_dir):
    """Build and save masks for one .czi, then return the analysis bundle."""
    filename = os.path.basename(filepath)
    stem = os.path.splitext(filename)[0]
    print(f"\nProcessing: {filename}")

    ch = load_four_channels(filepath)
    if ch is None:
        print("  Skipping (load error)."); return None
    raw = ch
    norm = {k: normalize_image(v) for k, v in ch.items()}  # min-max for inclusion
    seg = {k: pct_norm(v) for k, v in ch.items()}  # percentile for segmentation

    # Segmentation
    if USE_CELLPOSE:
        cyto3_cells = segment_whole_cells_cellpose(seg)  # cyto3 cells
        nucleus_mask, labeled_nuclei = segment_nucleus(seg)  # nucleus mask + labels
        labeled_cells = fill_missed_cells_from_nuclei(
            cyto3_cells, labeled_nuclei, cyto_composite(seg))
    else:
        cyto3_cells = None
        labeled_cells = segment_cells_intensity(raw["green"], raw["blue"])
        nucleus_mask = make_nucleus_mask(raw["blue"])
    if labeled_cells is None or labeled_cells.max() == 0:
        print("  No cells found, skipping."); return None

    # Green-positive cells
    green_positive_ids, fills = classify_green_positive(labeled_cells, seg["green"])
    print(f"    Green-positive cells: {len(green_positive_ids)} / {labeled_cells.max()} "
          f"(fill >= {GREEN_FILL_FRAC})")

    # Inclusion masks (use min-max `norm`)
    green_dots = detect_green_dots(norm["green"], labeled_cells)
    red_mask = detect_bright_puncta(norm["red"], RED_TOPHAT_RADIUS, RED_PCTL_THRESH, RED_MIN_SIZE, RED_MAX_SIZE)
    orange_raw = detect_bright_puncta(norm["orange"], ORANGE_TOPHAT_RADIUS, ORANGE_PCTL_THRESH, ORANGE_MIN_SIZE, ORANGE_MAX_SIZE)
    orange_mask, bleed_mask = remove_bleedthrough(orange_raw, red_mask, green_dots)

    masks = {
        "green_dots": green_dots,
        "red": red_mask,
        "orange": orange_mask,
        "bleed": bleed_mask,
    }

    # Save masks for this image
    img_mask_dir = os.path.join(mask_dir, stem)
    save_image_masks(img_mask_dir, labeled_cells, masks, green_positive_ids)
    print(f"    saved masks -> {img_mask_dir}")

    if SHOW_PLOTS:
        show_fourteen_panels(filename, raw, labeled_cells, green_positive_ids,
                             nucleus_mask, green_dots, red_mask, orange_raw,
                             orange_mask, bleed_mask, cyto3_cells)

    return {"labeled_cells": labeled_cells, "masks": masks,
            "raw": {"green": raw["green"]}, "green_positive_ids": green_positive_ids}


print("Per-image segmentation defined")

## Sequential driver

In [ ]:
def analyze_folder(image_folder):
    """Process one folder and write one CSV."""
    folder_tag = os.path.basename(os.path.normpath(image_folder))
    mask_dir = os.path.join(MASK_FOLDER, folder_tag)
    os.makedirs(mask_dir, exist_ok=True)

    image_files = sorted(glob.glob(os.path.join(image_folder, FILE_EXTENSION)))
    print(f"---------- {folder_tag}: {len(image_files)} image(s) ----------")

    rows = []
    for filepath in image_files:
        result = segment_one_image(filepath, mask_dir)
        if result is None:
            continue
        for cid in result["green_positive_ids"]:
            row = analyze_cell(cid, result["labeled_cells"], result["masks"], result["raw"])
            row["Filename"] = os.path.basename(filepath)
            rows.append(row)

    if rows:
        df = pd.DataFrame(rows)
        front = ["Filename", "Cell_ID"]
        df = df[front + [c for c in df.columns if c not in front]]
    else:
        df = pd.DataFrame()

    out_csv = os.path.join(RESULTS_FOLDER, f"{folder_tag}_{CSV_SUFFIX}")
    df.to_csv(out_csv, index=False)
    print(f"  -> {len(df)} green-positive cell row(s) saved to {out_csv}\n")
    return df


last_df = None
for image_folder in IMAGE_FOLDERS:
    folder_tag = os.path.basename(os.path.normpath(image_folder))
    if not os.path.isdir(image_folder):
        print(f"[skip] '{image_folder}' not found\n")
        continue
    print(f"{folder_tag}: START")
    last_df = analyze_folder(image_folder)

print("All folders processed"
      "masks in Results/masks/<folder>/<image>/.")
last_df.head() if last_df is not None else None

## Output summary

In [ ]:
# Summary of what the driver wrote.
csv_files = sorted(glob.glob(os.path.join(RESULTS_FOLDER, f"*_{CSV_SUFFIX}")))
print(f"{len(csv_files)} CSV file(s) in {RESULTS_FOLDER}/:")
total_rows = 0
for f in csv_files:
    n = len(pd.read_csv(f))
    total_rows += n
    print(f"  {os.path.basename(f):70s} {n:>4d} green-positive cell row(s)")
print(f"  {'TOTAL':70s} {total_rows:>4d}")